# Chest X-Ray Pneumonia Training

Trains the(**custom_cnn**, **densenet121**):

**cells**:
1. Confirm GPU
2. Clone repo, install deps
3. Mount Google Drive
4. Download dataset via **kagglehub**
5. Train **custom_cnn** (15 epochs)
6. Train **densenet121** (10 epochs)
7. Copy checkpoints + CSV logs to Drive


## 1. Confirm GPU

In [1]:
!nvidia-smi

Mon Apr 27 23:31:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone repo and install requirements

In [2]:
%cd /content
![ -d CS-171-Chest-X-Ray-Medical-Diagnosis ] || git clone https://github.com/jenilkathrotia/CS-171-Chest-X-Ray-Medical-Diagnosis.git
%cd /content/CS-171-Chest-X-Ray-Medical-Diagnosis
!git fetch --all
!git checkout part-2 || git checkout -b part-2 origin/part-2
!git pull origin part-2 || true
!pip install -q -r requirements.txt

/content
Cloning into 'CS-171-Chest-X-Ray-Medical-Diagnosis'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 56 (delta 9), reused 45 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 604.82 KiB | 2.29 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/CS-171-Chest-X-Ray-Medical-Diagnosis
Fetching origin
Branch 'part-2' set up to track remote branch 'part-2' from 'origin'.
Switched to a new branch 'part-2'
From https://github.com/jenilkathrotia/CS-171-Chest-X-Ray-Medical-Diagnosis
 * branch            part-2     -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.1 MB/s eta 0:00:00


## 3. Mount Google Drive

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/cs171_chest_xray'
os.makedirs(f'{DRIVE_OUT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_OUT}/logs', exist_ok=True)
print('Drive output dir:', DRIVE_OUT)

Mounted at /content/drive
Drive output dir: /content/drive/MyDrive/cs171_chest_xray


## 4. Download dataset via kagglehub

`kagglehub` will cache the dataset; the resulting path contains a `chest_xray/` folder with `train/`, `val/`, `test/`.

In [8]:
import kagglehub, os

raw_path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print('kagglehub root:', raw_path)

# The dataset usually unpacks under .../chest-xray-pneumonia/chest_xray.
candidates = [
    os.path.join(raw_path, 'chest_xray'),
    raw_path,
]
DATA_DIR = next(p for p in candidates if os.path.isdir(os.path.join(p, 'train')))
print('DATA_DIR:', DATA_DIR)
print('classes:', sorted(os.listdir(os.path.join(DATA_DIR, 'train'))))

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
kagglehub root: /kaggle/input/chest-xray-pneumonia
DATA_DIR: /kaggle/input/chest-xray-pneumonia/chest_xray
classes: ['NORMAL', 'PNEUMONIA']


## 5. Train Custom CNN (15 epochs)

In [9]:
import sys
if '/content/CS-171-Chest-X-Ray-Medical-Diagnosis' not in sys.path:
    sys.path.insert(0, '/content/CS-171-Chest-X-Ray-Medical-Diagnosis')

from src.train import train

custom_results = train(
    model_name='custom_cnn',
    data_dir=DATA_DIR,
    epochs=15,
    lr=1e-4,
    batch_size=32,
)
print(custom_results['final_epoch'])
print('checkpoint:', custom_results['checkpoint_path'])

[train] device = cuda
[train] class weights = [1.944817304611206, 0.673032283782959]
epoch 01/15  train_loss=0.3555 train_acc=0.8322  |  val_loss=0.8823 val_acc=0.6250  lr=1.00e-04
  -> saved checkpoint: /content/CS-171-Chest-X-Ray-Medical-Diagnosis/results/checkpoints/custom_cnn_best.pt
epoch 02/15  train_loss=0.2672 train_acc=0.8884  |  val_loss=1.1991 val_acc=0.5625  lr=1.00e-04
epoch 03/15  train_loss=0.2429 train_acc=0.8980  |  val_loss=1.4054 val_acc=0.5625  lr=1.00e-04
epoch 04/15  train_loss=0.2220 train_acc=0.9112  |  val_loss=1.4353 val_acc=0.6250  lr=1.00e-04
epoch 05/15  train_loss=0.2220 train_acc=0.9099  |  val_loss=1.2602 val_acc=0.6250  lr=5.00e-05
epoch 06/15  train_loss=0.2101 train_acc=0.9162  |  val_loss=1.3682 val_acc=0.5625  lr=5.00e-05
epoch 07/15  train_loss=0.2021 train_acc=0.9176  |  val_loss=1.2863 val_acc=0.6875  lr=5.00e-05
epoch 08/15  train_loss=0.2100 train_acc=0.9168  |  val_loss=1.0037 val_acc=0.6875  lr=2.50e-05
epoch 09/15  train_loss=0.1975 train_ac

## 6. Train DenseNet121 (10 epochs)

In [10]:
densenet_results = train(
    model_name='densenet121',
    data_dir=DATA_DIR,
    epochs=10,
    lr=1e-4,
    batch_size=32,
)
print(densenet_results['final_epoch'])
print('checkpoint:', densenet_results['checkpoint_path'])

[train] device = cuda
[train] class weights = [1.944817304611206, 0.673032283782959]
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 108MB/s] 


epoch 01/10  train_loss=0.1567 train_acc=0.9429  |  val_loss=0.2152 val_acc=0.9375  lr=1.00e-04
  -> saved checkpoint: /content/CS-171-Chest-X-Ray-Medical-Diagnosis/results/checkpoints/densenet121_best.pt
epoch 02/10  train_loss=0.0761 train_acc=0.9726  |  val_loss=0.2166 val_acc=0.8750  lr=1.00e-04
epoch 03/10  train_loss=0.0653 train_acc=0.9755  |  val_loss=0.1197 val_acc=0.9375  lr=1.00e-04
  -> saved checkpoint: /content/CS-171-Chest-X-Ray-Medical-Diagnosis/results/checkpoints/densenet121_best.pt
epoch 04/10  train_loss=0.0582 train_acc=0.9770  |  val_loss=0.4367 val_acc=0.8125  lr=1.00e-04
epoch 05/10  train_loss=0.0532 train_acc=0.9803  |  val_loss=0.3488 val_acc=0.8750  lr=1.00e-04
epoch 06/10  train_loss=0.0434 train_acc=0.9845  |  val_loss=0.0856 val_acc=0.9375  lr=1.00e-04
  -> saved checkpoint: /content/CS-171-Chest-X-Ray-Medical-Diagnosis/results/checkpoints/densenet121_best.pt
epoch 07/10  train_loss=0.0416 train_acc=0.9849  |  val_loss=1.3110 val_acc=0.6875  lr=1.00e-04
e

## 7. Copy checkpoints and CSV logs to Drive

In [12]:
import shutil

for name in ['custom_cnn', 'densenet121']:
    ckpt_src = f'results/checkpoints/{name}_best.pt'
    log_src = f'results/logs/{name}.csv'
    if os.path.exists(ckpt_src):
        shutil.copy2(ckpt_src, f'{DRIVE_OUT}/checkpoints/{name}_best.pt')
    if os.path.exists(log_src):
        shutil.copy2(log_src, f'{DRIVE_OUT}/logs/{name}.csv')

print('Drive contents:')
!ls -la {DRIVE_OUT}/checkpoints {DRIVE_OUT}/logs

Drive contents:
/content/drive/MyDrive/cs171_chest_xray/checkpoints:
total 28502
-rw------- 1 root root   750343 Apr 27 23:39 custom_cnn_best.pt
-rw------- 1 root root 28435350 Apr 28 00:10 densenet121_best.pt

/content/drive/MyDrive/cs171_chest_xray/logs:
total 3
-rw------- 1 root root 1165 Apr 27 23:59 custom_cnn.csv
-rw------- 1 root root  807 Apr 28 00:17 densenet121.csv
